# Homework 1 - 3008ICT

 ### No submission is required.


In [169]:
import torch
from jax import numpy as jnp
from flax import nnx
import time
import numpy as np


Rngs = nnx.Rngs(0)

## 1. Speedtest for vectorization

Your goal is to measure the speed of linear algebra operations for different levels of vectorization. 
1. Construct two matrices $A$ and $B$ with Gaussian random entries of size $4096 \times 4096$. 
1. Compute $C = A B$ using matrix-matrix operations and report the time. 
1. Compute $C = A B$, treating $A$ as a matrix but computing the result for each column of $B$ one at a time. Report the time.
1. Compute $C = A B$, treating $A$ and $B$ as collections of vectors. Report the time.
1. Optional - what changes if you execute this on a GPU?

In [ ]:
## 1

size = 3

A = torch.randn((size, size))
B = torch.randn((size, size))

## 2

C = A @ B

## 3

C2 = torch.zeros_like(A)

for i, col in enumerate(B.T):
    C2[:, i] = A @ col

assert (C - C2).abs().max() < 1e-4

## 4

C3 = torch.zeros_like(A)

for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        C3[i,j] = A[i,:] @ B[:,j]

assert (C - C3).abs().max() < 1e-4

## 5

# 2 will get faster.
# 3 and 4 will probably get slower

Pytorch took 0.00027489662170410156s
Pytorch one-at-a-time took 0.0008795261383056641s


## 2. NDArray and NumPy 

Your goal is to measure the speed penalty between Pytorch and Python when converting data between both. We are going to do this as follows:

1. Create two Gaussian random matrices $A, B$ of size $4096 \times 4096$ in PyTorch. 
1. Compute a vector $\mathbf{c} \in \mathbb{R}^{4096}$ where $c_i = \|A B_{i\cdot}\|^2$ where $\mathbf{c}$ is a **NumPy** vector.

To see the difference in speed due to Python perform the following two experiments and measure the time:

1. Compute $\|A B_{i\cdot}\|^2$ one at a time and assign its outcome to $\mathbf{c}_i$ directly.
1. Use an intermediate storage vector $\mathbf{d}$ in NDArray for assignments and copy to NumPy at the end.

In [179]:
## NEED TO CHECK FORMATTING

A = torch.randn((4096, 4096))
B = torch.randn((4096, 4096))

time1 = time.time()
C = np.zeros((4096))

for i in range(C.shape[0]):
    C[i] = (A @ B[i, :]).pow(2).sum()
time1s = time.time()

time2 = time.time()
C2 = torch.zeros((4096))
for i in range(C.shape[0]):
    C2[i] = (A @ B[i, :]).pow(2).sum()
C3 = C2.numpy()
time2s = time.time()

print(f"1: {time1s-time1}, 2: {time2s-time2}")


1: 6.230448961257935, 2: 5.6406097412109375


## 3. Memory efficient computation

We want to compute $C \leftarrow A \cdot B + C$, where $A, B$ and $C$ are all matrices. Implement this in the most memory efficient manner. Pay attention to the following two things:

1. Do not allocate new memory for the new value of $C$.
1. Do not allocate new memory for intermediate results if possible.

In [181]:
A = torch.randn((4096, 4096))
B = torch.randn((4096, 4096))
C = torch.randn((4096, 4096))

print(C[0,0])
C.addmm_(A, B)
print(C[0,0])


tensor(-1.1536)
tensor(23.0065)


## 4. Broadcast Operations

In order to perform polynomial fitting we want to compute a design matrix $A$ with 

$$A_{ij} = x_i^j$$

Our goal is to implement this **without a single for loop** entirely using vectorization and broadcast. 
    Here $1 \leq j \leq 20$ and $x = \{-10, -9.9, \ldots 10\}$. Implement code that generates such a matrix. 
    Note that $x_i^j$ is the result of $x_i$ to the power of $j$. 


In [193]:
power = torch.arange(1, 21)
X = torch.arange(-10, 10.1, 0.1).reshape((-1,1))
A = X**power
print(A[0])

tensor([-1.0000e+01,  1.0000e+02, -1.0000e+03,  1.0000e+04, -1.0000e+05,
         1.0000e+06, -1.0000e+07,  1.0000e+08, -1.0000e+09,  1.0000e+10,
        -1.0000e+11,  1.0000e+12, -1.0000e+13,  1.0000e+14, -1.0000e+15,
         1.0000e+16, -1.0000e+17,  1.0000e+18, -1.0000e+19,  1.0000e+20])
